# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their IDs, fields (columns), and their `@id`s.

In [ ]:
# List all record sets in the dataset
record_sets = metadata.record_sets
print("Record Sets (@id and name):")
for rs in record_sets:
    print(f"  @id: {rs.id}  -  Name: {rs.name}")

# For each record set, print its fields and their @id
for rs in record_sets:
    print(f"\nFields for record set '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"  Field Name: {field.name:30} | @id: {field.id:40} | DataType: {field.data_type}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
dataframes = dict()

for rs in metadata.record_sets:
    print(f"Loading records for record set: {rs.name} (@id: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# For demonstration, select the first record set
main_record_set_id = metadata.record_sets[0].id  # Choose first as an example
print(f"\nMain record set selected for further analysis: {main_record_set_id}")
print("Available columns:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll choose a numeric field (e.g., `cr:MW` for molecular weight if present) for illustration. All fields/columns are referenced by their `@id` per the Croissant model, and variables are used.

In [ ]:
# Select a numeric field for analysis, e.g., 'cr:MW' (molecular weight) if available
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Try to automatically pick a numeric field from the first record set fields
numeric_field = None
for field in metadata.get_record_set(record_set_id).fields:
    if field.data_type and (str(field.data_type).lower() in ['number','integer','float']):
        numeric_field = field.id
        break

if numeric_field is None:
    # If not found, use a common field name seen in proteomics datasets
    possible_fields = ['cr:MW', 'cr:peptide_count', 'cr:coverage', 'cr:peptides', 'cr:abundance']
    for col in df.columns:
        if col in possible_fields:
            numeric_field = col
            break

if numeric_field is None:
    numeric_field = df.select_dtypes('number').columns[0]

print(f"Using numeric field for analysis: {numeric_field}")

# Drop rows with missing values in the numeric field
df_numeric = df.dropna(subset=[numeric_field])

threshold = df_numeric[numeric_field].mean()  # Filter above mean as threshold example

filtered_df = df_numeric[df_numeric[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalize the field (z-score)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by another available field, e.g., 'cr:description' if present
group_field = None
for candidate in ["cr:description", "cr:accession", "cr:peptide_count", "cr:peptides", "cr:sample_id"]:
    if candidate in df.columns and candidate != numeric_field:
        group_field = candidate
        break

if group_field is not None:
    print(f"\nGrouping data by {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and relationships if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df_numeric.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(df_numeric[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If another numeric field exists, plot scatter
    numeric_cols = df_numeric.select_dtypes('number').columns.tolist()
    if len(numeric_cols) > 1:
        plt.figure(figsize=(7,6))
        other_field = [col for col in numeric_cols if col != numeric_field][0]
        sns.scatterplot(x=df_numeric[numeric_field], y=df_numeric[other_field])
        plt.xlabel(numeric_field)
        plt.ylabel(other_field)
        plt.title(f"Scatter plot of {numeric_field} vs {other_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, analyze, and visualize the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

**Key findings:**
- The dataset contains detailed mass spectrometry proteomics records, including molecular weight (MW), peptide counts, protein descriptions, and more, each referenced by their `@id` as specified in the Croissant schema.
- We demonstrated extracting records by their record set `@id`, selecting and analyzing numeric fields, filtering and normalizing, and visualizing distributions.

Feel free to adapt this notebook for deeper biological, statistical, or machine learning analyses. Always reference fields and record sets by their `@id` for precise schema-compliant data handling.